In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv("Housing.csv")

# Separate features and target
X = df.drop("price", axis=1)
y = df["price"]

# Encode categorical columns
X = pd.get_dummies(X, dtype=int, drop_first=True)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Evaluate model
y_pred = model.predict(X_test_scaled)

print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

# Save model, scaler, and columns
with open("house_price_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("house_price_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("house_price_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Model files saved successfully!")

MSE: 1754318687330.669
MAE: 970043.4039201644
RMSE: 1324506.9600914405
R2 Score: 0.6529242642153175
Model files saved successfully!


In [3]:
import pickle
import pandas as pd
import gradio as gr

# Load saved files
with open("house_price_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("house_price_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("house_price_columns.pkl", "rb") as f:
    columns = pickle.load(f)


def predict_price(
    area, bedrooms, bathrooms, stories, mainroad,
    guestroom, basement, hotwaterheating,
    airconditioning, parking, prefarea, furnishingstatus
):
    # Create input DataFrame
    data = pd.DataFrame([{
        "area": area,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "stories": stories,
        "mainroad": mainroad,
        "guestroom": guestroom,
        "basement": basement,
        "hotwaterheating": hotwaterheating,
        "airconditioning": airconditioning,
        "parking": parking,
        "prefarea": prefarea,
        "furnishingstatus": furnishingstatus
    }])

    # Apply one-hot encoding
    data = pd.get_dummies(data, dtype=int, drop_first=True)

    # Match training columns
    data = data.reindex(columns=columns, fill_value=0)

    # Scale input
    data_scaled = scaler.transform(data)

    # Predict price
    prediction = model.predict(data_scaled)[0]

    return f"Estimated House Price: {prediction:,.0f}"


app = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Number(label="Area", value=5000),
        gr.Number(label="Bedrooms", value=3),
        gr.Number(label="Bathrooms", value=2),
        gr.Number(label="Stories", value=2),
        gr.Radio(["yes", "no"], label="Main Road", value="yes"),
        gr.Radio(["yes", "no"], label="Guestroom", value="no"),
        gr.Radio(["yes", "no"], label="Basement", value="no"),
        gr.Radio(["yes", "no"], label="Hot Water Heating", value="no"),
        gr.Radio(["yes", "no"], label="Air Conditioning", value="yes"),
        gr.Number(label="Parking Spaces", value=1),
        gr.Radio(["yes", "no"], label="Preferred Area", value="yes"),
        gr.Radio(
            ["furnished", "semi-furnished", "unfurnished"],
            label="Furnishing Status",
            value="furnished"
        )
    ],
    outputs=gr.Textbox(label="Predicted Price"),
    title="House Price Prediction System",
    description="Enter house details to predict its price."
)

if __name__ == "__main__":
    app.launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [4]:
import os

os.makedirs("templates", exist_ok=True)

html_code = """
<!DOCTYPE html>
<html>
<head>
    <title>House Price Prediction</title>
    <style>
        body {
            font-family: Arial;
            background: #eef2f7;
            padding: 25px;
        }
        .container {
            max-width: 650px;
            margin: auto;
            background: white;
            padding: 25px;
            border-radius: 12px;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 15px;
        }
        input, select, button {
            box-sizing: border-box;
            width: 100%;
            padding: 10px;
            margin-top: 5px;
        }
        button {
            background: #1d4ed8;
            color: white;
            border: none;
            cursor: pointer;
            margin-top: 20px;
        }
        .result {
            background: #dcfce7;
            padding: 15px;
            margin-top: 20px;
            text-align: center;
        }
    </style>
</head>
<body>
<div class="container">
    <h1>House Price Prediction</h1>

    <form method="POST">
        <div class="grid">
            <div>Area
                <input type="number" name="area" min="1" required>
            </div>
            <div>Bedrooms
                <input type="number" name="bedrooms" min="0" required>
            </div>
            <div>Bathrooms
                <input type="number" name="bathrooms" min="0" required>
            </div>
            <div>Stories
                <input type="number" name="stories" min="1" required>
            </div>
            <div>Main Road
                <select name="mainroad">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Guestroom
                <select name="guestroom">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Basement
                <select name="basement">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Hot Water Heating
                <select name="hotwaterheating">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Air Conditioning
                <select name="airconditioning">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Parking Spaces
                <input type="number" name="parking" min="0" required>
            </div>
            <div>Preferred Area
                <select name="prefarea">
                    <option>yes</option><option>no</option>
                </select>
            </div>
            <div>Furnishing Status
                <select name="furnishingstatus">
                    <option>furnished</option>
                    <option>semi-furnished</option>
                    <option>unfurnished</option>
                </select>
            </div>
        </div>
        <button type="submit">Predict Price</button>
    </form>

    {% if prediction is not none %}
    <div class="result">
        <h2>Estimated House Price</h2>
        <h2>{{ prediction }}</h2>
    </div>
    {% endif %}
</div>
</body>
</html>
"""

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_code)

print("Flask HTML template created!")

# Code End


Flask HTML template created!


In [ ]:
import pandas as pd
import traceback
from flask import Flask, render_template, request

# Create Flask application
flask_app = Flask("house_price_app")


# Home page and prediction route
@flask_app.route("/", methods=["GET", "POST"])
def home():

    prediction = None
    error = None

    if request.method == "POST":
        try:
            # Get house details from HTML form
            input_data = {
                "area": float(request.form["area"]),
                "bedrooms": int(request.form["bedrooms"]),
                "bathrooms": int(request.form["bathrooms"]),
                "stories": int(request.form["stories"]),
                "mainroad": request.form["mainroad"],
                "guestroom": request.form["guestroom"],
                "basement": request.form["basement"],
                "hotwaterheating": request.form["hotwaterheating"],
                "airconditioning": request.form["airconditioning"],
                "parking": int(request.form["parking"]),
                "prefarea": request.form["prefarea"],
                "furnishingstatus": request.form["furnishingstatus"]
            }

            # Convert input into a DataFrame
            data = pd.DataFrame([input_data])

            # Encode categorical features
            data = pd.get_dummies(data, dtype=int)

            # Match training feature columns and their order
            data = data.reindex(
                columns=X.columns,
                fill_value=0
            )

            # Scale the input data
            data_scaled = scaler.transform(data)

            # Predict house price
            result = model.predict(data_scaled)[0]

            # Format predicted price
            prediction = f"{result:,.0f}"

        except Exception:
            # Print the detailed error in Jupyter
            error = traceback.format_exc()
            print("Prediction Error:\n", error)

    # Display the HTML page
    return render_template(
        "index.html",
        prediction=prediction,
        error=error
    )


# Start Flask server
if __name__ == "__main__":
    flask_app.run(
        host="127.0.0.1",
        port=5001,
        debug=True,
        use_reloader=False
    )

 * Serving Flask app 'house_price_app'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [11/Oct/2026 22:24:55] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Oct/2026 22:25:05] "POST / HTTP/1.1" 200 -
